# CDC-CP P2 sweep: fight the crop shortcut, then score with crop-balanced prototypes

The model tends to guess the disease from the crop. On unseen pairs, most wrong disease guesses are a disease the true crop had in training. This notebook trains four ConvNeXt-Tiny runs that try to fix it, and measures the **shortcut rate** for every run.

All runs use ConvNeXt-Tiny, head 256 with GELU, batch 16 x 2 views, 10 epochs (4,310 applied updates), backbone LR 1e-4, heads 1e-3. Only the P2 loss changes.

| Config | What it tests |
|---|---|
| `p2c_ctl` | Control: SupCon off |
| `p2c_plain` | Plain SupCon on crop and disease |
| `p2c_cross` | Cross-crop SupCon for disease (positives from other crops) with a small per-pair memory |
| `p2c_cross_spy` | `p2c_cross` plus a crop spy that is trained against the disease embedding |

| Phase | What runs | Runs |
|---|---|---|
| **A** | The four configs above, seed 42 | 4 |
| **B** | Seed 43 of the winner and of `p2c_ctl`. Winner = the config with the best unseen disease Top-1 whose keep-seen H is at most 5 points below the control's. The control competes too; if it wins, only `p2c_ctl` runs | 2 (1 if the control wins) |
| **C** | `scripts/p3_eval.py` on the last epoch of both seeds of the winner and of `p2c_ctl` | 4 evaluations (2 if the control wins) |

**Reading the result.** Calibrated H alone flatters a model that knows the crop (each crop has one unseen pair), so the winner is picked on unseen disease Top-1 with H as a guard. `scripts/calib_board.py` gives the keep-seen board. A second table gives, per run, the mean over the last 3 epochs of `unseen_shortcut_rate` (lower is better), `unseen_oracle_disease_top1` and `unseen_disease_top1` from `history.csv`.

**Setup (once):** inputs are the code dataset and the data dataset. Then **Accelerator -> GPU T4 x2** and **Internet -> On**.

**Run unattended:** **Save Version -> Save & Run All (Commit)**. Two runs train at once, one per GPU.

**If it stops early:** Add Input -> this notebook's last version (its output), then Save & Run All again. Finished runs are skipped and an interrupted run resumes.

Only `DPD_Small_train.csv` and its 8,000 images are read. The official test files are never opened.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE    = False   # True: 1 epoch of every phase-A config, to check everything builds on the GPU
TOL      = 3.0     # keep-seen: seen accuracy may fall at most this many points below raw
WORKERS  = 2       # data-loader workers per run (2 runs share Kaggle's 4 CPU cores)

CONFIGS = ['p2c_ctl', 'p2c_plain', 'p2c_cross', 'p2c_cross_spy']   # configs/<name>.yaml; the run folder is <name>_seed<seed>
CONTROL = 'p2c_ctl'
GUARD_H = 5.0     # the winner's keep-seen H may be at most this many points below the control's (about the seed noise)

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/p2_smoke' if SMOKE else '/kaggle/working/runs')
CODE_DIR  = Path('/tmp/cdc-cp')
DATA_DIR  = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'scripts' / 'p3_eval.py').is_file(), 'old code dataset: upload the new cdc-cp_code.zip (it must contain scripts/p3_eval.py)'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
missing_cfg = [c for c in CONFIGS if not (CODE_DIR / 'configs' / f'{c}.yaml').is_file()]
assert not missing_cfg, f'code dataset lacks configs {missing_cfg}: upload the new code zip'
print('code from', found or code_zip)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from an earlier version's output

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
restored = []
for last in INPUT.rglob('p2c_*_seed*/last.pt'):
    src, dst = last.parent, RUNS_DIR / last.parent.name
    old = dst / 'last.pt'
    if not old.exists() or torch.load(old, map_location='cpu', weights_only=False)['applied'] < torch.load(last, map_location='cpu', weights_only=False)['applied']:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        restored.append(dst.name)
print('restored:', restored or 'nothing (first version)')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. A run's log lines are printed as it trains, prefixed with its name. When a run finishes, the scoreboard can read it. Only its last 3 epoch checkpoints are kept, which stays inside Kaggle's 20 GB output limit.

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']


def run_name(config, seed):
    return f'{config}_seed{seed}'


def job(config, seed=42):
    args = [sys.executable, 'scripts/train.py', f'configs/{config}.yaml', *COMMON, f'seed={seed}', '--resume']
    if SMOKE:
        args += ['--max-epochs', '1']
    return run_name(config, seed), args


def prune(run_dir, keep=3):
    pts = sorted(run_dir.glob('epoch_*.pt'))
    for p in pts[:-keep]:
        p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed


def board(*patterns):
    sh(sys.executable, 'scripts/calib_board.py', RUNS_DIR, '--tol', TOL, '--match', *patterns)
    return pd.read_csv(RUNS_DIR / 'calib_board.csv')


def ranked_configs(df):
    """Configs by mean keep-seen H, best first."""
    return df.groupby('config')['H keep'].mean().sort_values(ascending=False)


def pick_winner(df):
    """Config with the best unseen disease Top-1, among those whose keep-seen H is within GUARD_H of the control.

    The control competes too, so a variant wins only if it beats the control on unseen disease Top-1.
    Unseen disease Top-1 is used because calibrated H rewards knowing the crop: each crop has one unseen pair, so a
    right crop plus the seen-pair bias already gives the right unseen pair.
    """
    by = df.groupby('config')[['unseen disease', 'H keep']].mean()
    if CONTROL not in by.index:
        raise RuntimeError(f'the control {CONTROL} has no finished run on the board: there is nothing to compare against')
    floor = by.loc[CONTROL, 'H keep'] - GUARD_H
    ok = by[by['H keep'] >= floor].assign(variant=lambda d: d.index != CONTROL)
    ok = ok.sort_values(['unseen disease', 'variant'], ascending=[False, True])   # the control wins a tie
    print(by.round(2).sort_values('unseen disease', ascending=False))
    print(f'H floor {floor:.1f} (control H keep minus {GUARD_H})')
    if ok.index[0] == CONTROL:
        print(f'No variant beats the control on unseen disease Top-1 within the H guard: phase B and C use {CONTROL} only.')
    return ok.index[0]


SHORTCUT_COLS = ['unseen_shortcut_rate', 'unseen_shortcut_chance', 'unseen_oracle_disease_top1', 'unseen_disease_top1']


def shortcut_table(names):
    """Mean of each run's last 3 epochs for the shortcut columns of history.csv."""
    rows = []
    for n in names:
        path = RUNS_DIR / n / 'history.csv'
        if not path.is_file():
            continue
        h = pd.read_csv(path).tail(3)
        rows.append({'run': n, 'epochs': len(h), **{c: h[c].mean() for c in SHORTCUT_COLS if c in h}})
    return pd.DataFrame(rows).round(2)

## 7. Phase A: the four P2 configs, seed 42

In [ ]:
FAILED_A = run_queue([job(c) for c in CONFIGS])
if SMOKE:
    assert not FAILED_A, f'SMOKE failed for {FAILED_A}: read their logs in {RUNS_DIR} before the full run'
    print('SMOKE run finished: every config trains on this GPU. Set SMOKE = False and Save & Run All (Commit).')
else:
    assert run_name(CONTROL, 42) not in FAILED_A, f'the control {CONTROL} failed: there is nothing to compare against'
    WINNER = pick_winner(board('p2c_*_seed42'))
    print('winner (unseen disease Top-1, H-guarded):', WINNER)

## 8. Phase B: second seed for the winner and the control

In [ ]:
if not SMOKE:
    FINALISTS = list(dict.fromkeys((WINNER, CONTROL)))   # one entry when the control is the winner
    FAILED_B = run_queue([job(c, seed=43) for c in FINALISTS])

## 9. Phase C: prototype scoring on the last epoch

In [ ]:
if not SMOKE:
    import yaml
    for cfg_name in FINALISTS:
        for seed in (42, 43):
            name = run_name(cfg_name, seed)
            run_dir = RUNS_DIR / name
            history = run_dir / 'history.csv'
            last = int(pd.read_csv(history).epoch.max()) if history.exists() else 0
            epochs = yaml.safe_load((CODE_DIR / 'configs' / f'{cfg_name}.yaml').read_text(encoding='utf-8'))['epochs']
            if name in FAILED_B or last < epochs:
                print(f'SKIP P3 for {name}: the run failed or stopped at epoch {last} of {epochs}', flush=True)
                continue
            sh(sys.executable, 'scripts/p3_eval.py', run_dir, '--epoch', last, '--tol', TOL,
               f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}')

## 10. Final scoreboard and shortcut table
The calibrated board is saved as `calib_board.md`, `calib_board.csv` and `gamma_sensitivity.csv` in this version's output. The shortcut table is the mean of each run's last 3 epochs, read from `history.csv`. A lower `unseen_shortcut_rate` means less crop shortcut. `unseen_shortcut_chance` is the rate expected if the wrong guess were random.

In [ ]:
if not SMOKE:
    final = board('p2c_*')
    from IPython.display import Markdown, display
    display(Markdown((RUNS_DIR / 'calib_board.md').read_text(encoding='utf-8')))
    names = sorted(d.name for d in RUNS_DIR.glob('p2c_*_seed*') if d.is_dir())
    table = shortcut_table(names)
    table.to_csv(RUNS_DIR / 'shortcut_table.csv', index=False)
    display(table)
    for p in sorted(RUNS_DIR.glob('*/p3_board_ep*.md')):
        display(Markdown(p.read_text(encoding='utf-8')))

## 11. Curves

In [ ]:
if not SMOKE:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    for n in [n for n in names if (RUNS_DIR / n / 'history.csv').is_file()]:
        h = pd.read_csv(RUNS_DIR / n / 'history.csv')
        line, = axes[0].plot(h.epoch, h.A_S, label=n)
        axes[1].plot(h.epoch, h.unseen_disease_top1, color=line.get_color(), label=n)
        axes[2].plot(h.epoch, h.unseen_shortcut_rate, color=line.get_color(), label=n)
    for ax, t in zip(axes, ['Seen pairs Top-1 (raw)', 'Unseen disease Top-1', 'Unseen shortcut rate (lower is better)']):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()